In [ ]:
// ============================================================
// MODIS AQUA DAILY LST
// MONTHLY MEAN LST
// ============================================================

// Replace with your state asset
var state = ee.FeatureCollection("users/your_username/state");

var geometry = state.geometry().simplify(1000);

Map.centerObject(state, 7);

// ============================================================
// AQUA DAILY LST
// ============================================================

var modis = ee.ImageCollection("MODIS/061/MYD11A1");

// ============================================================
// QC MASK
// ============================================================

function maskLST(img) {

  var qc = img.select('QC_Day');

  var good = qc
    .bitwiseAnd(3)
    .eq(0);

  return img
    .updateMask(good)
    .select('LST_Day_1km')
    .multiply(0.02)
    .subtract(273.15)
    .copyProperties(img, ['system:time_start']);
}

// ============================================================
// EXPORT FUNCTION
// ============================================================

function exportMonth(year, month) {

  var start = ee.Date.fromYMD(year, month, 1);
  var end = start.advance(1, 'month');

  var monthly = modis
    .filterDate(start, end)
    .map(maskLST)
    .mean()
    .clip(state);

  var mm = month < 10 ? '0' + month : month;

  var filename =
    'LST_STATE_' +
    year +
    '_' +
    mm;

  Export.image.toDrive({
    image: monthly,
    description: filename,
    fileNamePrefix: filename,
    folder: 'LST',
    region: geometry.bounds(),
    scale: 1000,
    crs: 'EPSG:4326',
    maxPixels: 1e13
  });
}

// ============================================================
// EXAMPLE: JAN 2025
// ============================================================

exportMonth(2025, 1);

// ============================================================
// PREVIEW
// ============================================================

var preview = modis
  .filterDate('2025-01-01', '2025-02-01')
  .map(maskLST)
  .mean()
  .clip(state);

Map.addLayer(
  preview,
  {
    min: 15,
    max: 45
  },
  'Jan 2025 LST'
);